In [2]:
# Colab-ready: Full standard U-Net + SAVE EVERY KERNEL OUTPUT + random image selection
!pip install -q torchvision tqdm

import os
import math
import random
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 128
BATCH_SIZE = 1
ROOT = "data"
OUTDIR = "unet_all_kernels_outputs"
os.makedirs(OUTDIR, exist_ok=True)
plt.rcParams['figure.figsize'] = (10,6)

############################################
#  U-N E T   A R C H I T E C T U R E
############################################
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu2 = nn.ReLU(inplace=True)

    def forward(self, x, collect=None, name_prefix=""):
        x = self.relu1(self.bn1(self.conv1(x)))
        if collect is not None:
            collect.append((f"{name_prefix}_conv1", x.detach().cpu()))
        x = self.relu2(self.bn2(self.conv2(x)))
        if collect is not None:
            collect.append((f"{name_prefix}_conv2", x.detach().cpu()))
        return x

class UNetFull(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, features=[64,128,256,512,1024]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.pools = nn.ModuleList()
        prev = in_ch
        for f in features:
            self.downs.append(DoubleConv(prev, f))
            self.pools.append(nn.MaxPool2d(2))
            prev = f

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        rev = list(reversed(features))
        self.up_transposes = nn.ModuleList()
        self.up_convs = nn.ModuleList()

        up_in = features[-1]*2
        for f in rev:
            self.up_transposes.append(nn.ConvTranspose2d(up_in, f, 2, stride=2))
            self.up_convs.append(DoubleConv(f*2, f))
            up_in = f

        self.final = nn.Conv2d(features[0], out_ch, 1)

    def forward(self, x, collect_maps=False):
        skips = []
        collected = [] if collect_maps else None

        for i, down in enumerate(self.downs):
            x = down(x, collect=collected, name_prefix=f"down{i+1}")
            skips.append(x)
            x = self.pools[i](x)
            if collect_maps:
                collected.append((f"after_pool{i+1}", x.detach().cpu()))

        x = self.bottleneck(x, collect=collected, name_prefix="bottleneck")

        for i in range(len(self.up_transposes)):
            x = self.up_transposes[i](x)
            if collect_maps:
                collected.append((f"up_transpose_{i+1}", x.detach().cpu()))

            skip = skips[-(i+1)]
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear')
            x = torch.cat([skip, x], dim=1)
            x = self.up_convs[i](x, collect=collected, name_prefix=f"up{i+1}")

        out = self.final(x)
        if collect_maps:
            collected.append(("final", out.detach().cpu()))
            return out, collected
        return out


############################################
#   RANDOM SAMPLE IMAGE LOADER  (fixed)
############################################
transform_img = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
])

def get_random_sample_image():
    try:
        ds = OxfordIIITPet(root=ROOT, split="trainval", target_types="segmentation", download=True)
        idx = random.randint(0, len(ds)-1)    # ←← RANDOM INDEX
        pil_img, _ = ds[idx]
        print(f"Random dataset index selected: {idx}")
        pil_img = pil_img.convert("RGB")
        img = transform_img(pil_img).unsqueeze(0)
        return img
    except Exception as e:
        print("Dataset failed, using synthetic image:", e)
        arr = np.random.randint(0,255,(IMAGE_SIZE,IMAGE_SIZE,3),dtype=np.uint8)
        from PIL import Image
        return transform_img(Image.fromarray(arr)).unsqueeze(0)


############################################
#   SAVE FEATURE MAP CHANNELS TO FILES
############################################
def normalize_np(t):
    t = t.copy()
    t -= t.min()
    if t.max() > 0:
        t /= t.max()
    return t

def save_all_channels(tensor4, layer_name, save_dir=OUTDIR):
    layer_folder = os.path.join(save_dir, layer_name)
    os.makedirs(layer_folder, exist_ok=True)
    C = tensor4.shape[1]

    for ch in range(C):
        arr = normalize_np(tensor4[0, ch].numpy())
        plt.imsave(os.path.join(layer_folder, f"ch_{ch}.png"), arr, cmap='gray')


############################################
#   RUN MODEL + SAVE ALL FEATURE MAPS
############################################
model = UNetFull().to(DEVICE)
model.eval()

img = get_random_sample_image().to(DEVICE)

with torch.no_grad():
    out, collected = model(img, collect_maps=True)

print("\nCollected layers:", len(collected))

# SAVE ALL FEATURE MAPS
for (layer_name, fmap) in collected:
    print(f"Saving layer: {layer_name}, shape={tuple(fmap.shape)}")
    save_all_channels(fmap, layer_name)

# Save input and final output
plt.imsave(os.path.join(OUTDIR, "input_image.png"),
           img[0].permute(1,2,0).cpu().numpy())

plt.imsave(os.path.join(OUTDIR, "final_output.png"),
           normalize_np(out[0,0].cpu().numpy()),
           cmap='gray')

print("\n✅ ALL DONE — all kernel/channel outputs saved in:", OUTDIR)


Random dataset index selected: 2369

Collected layers: 33
Saving layer: down1_conv1, shape=(1, 64, 128, 128)
Saving layer: down1_conv2, shape=(1, 64, 128, 128)
Saving layer: after_pool1, shape=(1, 64, 64, 64)
Saving layer: down2_conv1, shape=(1, 128, 64, 64)
Saving layer: down2_conv2, shape=(1, 128, 64, 64)
Saving layer: after_pool2, shape=(1, 128, 32, 32)
Saving layer: down3_conv1, shape=(1, 256, 32, 32)
Saving layer: down3_conv2, shape=(1, 256, 32, 32)
Saving layer: after_pool3, shape=(1, 256, 16, 16)
Saving layer: down4_conv1, shape=(1, 512, 16, 16)
Saving layer: down4_conv2, shape=(1, 512, 16, 16)
Saving layer: after_pool4, shape=(1, 512, 8, 8)
Saving layer: down5_conv1, shape=(1, 1024, 8, 8)
Saving layer: down5_conv2, shape=(1, 1024, 8, 8)
Saving layer: after_pool5, shape=(1, 1024, 4, 4)
Saving layer: bottleneck_conv1, shape=(1, 2048, 4, 4)
Saving layer: bottleneck_conv2, shape=(1, 2048, 4, 4)
Saving layer: up_transpose_1, shape=(1, 1024, 8, 8)
Saving layer: up1_conv1, shape=(1, 1

In [ ]:
import shutil
shutil.make_archive("unet_outputs_zip", "zip", "unet_all_kernels_outputs")
from google.colab import files
files.download("unet_outputs.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>